In [ ]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims
import numpy as np

In [ ]:
subject = "213421"
region = "CINGULATE"
side = "R"

In [ ]:
subject = "197550"
side = "R"
region = "CINGULATE" #"CINGULATE"

path_to_t1mri = f'/neurospin/dico/data/bv_databases/human/not_labeled/hcp/hcp/{subject}/t1mri/BL'

t1mri = a.loadObject(f'{path_to_t1mri}/{subject}.nii.gz') 
t1mri.loadReferentialFromHeader()

if True:
    # view an object in a 4 views block
    block = a.createWindowsBlock(2) # 2 columns
    w1 = a.createWindow("Axial", block=block)
    w2 = a.createWindow("Sagittal", block=block)
    w3 = a.createWindow("Coronal", block=block)
    w4 = a.createWindow("3D", block=block)
    t1mri.addInWindows([w1, w2, w3, w4])
    
    # Load an object
    Rwhite = a.loadObject(f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject}_Rwhite.gii')
    Lwhite = a.loadObject(f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject}_Lwhite.gii')
    Rwhite.loadReferentialFromHeader()
    Lwhite.loadReferentialFromHeader()


    w4.addObjects(Rwhite)
    w4.addObjects(Lwhite)

    Rsulci = a.loadObject(f'{path_to_t1mri}/default_analysis/folds/3.1/R{subject}.arg')
    Lsulci = a.loadObject(f'{path_to_t1mri}/default_analysis/folds/3.1/L{subject}.arg')
    Rsulci.loadReferentialFromHeader()
    Lsulci.loadReferentialFromHeader()


    #w3d.addObjects(Rsulci)
    #w3d.addObjects(Lsulci)

    w4.addObjects(Rsulci)
    w4.addObjects(Lsulci)

    w5 = a.createWindow("3D")
    w5.addObjects(Lwhite)
    w5.addObjects(Lsulci)


In [ ]:
# for UKB

ukb_from_TGCC = '/volatile/ad279118/Irene/MOSTEST/Results/ChampollionV0/ORBITAL_left/pattern_ISBI_2025'

list_subjects = ['sub-5627925',
 'sub-2714776',
 'sub-5439435',
 'sub-5577118',
 'sub-5702229',]

block = a.createWindowsBlock(5) # 5 columns
dic_windows = {}

for subject in list_subjects:
    dic_windows[f'w{subject}'] = a.createWindow("3D", block=block)

    dic_windows[f'white_{subject}'] = a.loadObject(f'{ukb_from_TGCC}/{subject}_{side}white.gii')
    dic_windows[f'white_{subject}'].loadReferentialFromHeader()

    dic_windows[f'sulci_{subject}'] = a.loadObject(f'{ukb_from_TGCC}/{side}{subject}.arg')
    dic_windows[f'sulci_{subject}'].loadReferentialFromHeader()

    dic_windows[f'w{subject}'].addObjects([dic_windows[f'white_{subject}'], dic_windows[f'sulci_{subject}']])

In [ ]:
#for HCP

list_subjects = [102816,
 103010,
 103111,
 103212,
 103414,
 103515,
 103818,
 104012,
 104416,
 104820,
 105014,
 105115,
 105216,
 105620,
 105923]
block = a.createWindowsBlock(5) # 5 columns
dic_windows = {}

for subject in list_subjects:
    path_to_t1mri = f'/neurospin/dico/data/bv_databases/human/not_labeled/hcp/hcp/{subject}/t1mri/BL'
    dic_windows[f'w{subject}'] = a.createWindow("3D", block=block)

    dic_windows[f'white_{subject}'] = a.loadObject(f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject}_{side}white.gii')
    dic_windows[f'white_{subject}'].loadReferentialFromHeader()

    dic_windows[f'sulci_{subject}'] = a.loadObject(f'{path_to_t1mri}/default_analysis/folds/3.1/{side}{subject}.arg')
    dic_windows[f'sulci_{subject}'].loadReferentialFromHeader()

    dic_windows[f'w{subject}'].addObjects([dic_windows[f'white_{subject}'], dic_windows[f'sulci_{subject}']])


In [ ]:
from soma import aims

def to_bucket(obj):
    if obj.type() == obj.BUCKET:
        return obj
    avol = a.toAimsObject(obj)
    c = aims.Converter(intype=avol, outtype=aims.BucketMap_VOID)
    abck = c(avol)
    bck = a.toAObject(abck)
    bck.releaseAppRef()
    return bck

id = "197550"
windoforcrop = a.createWindow("3D")
mm_skeleton_path = f"/neurospin/dico/data/deep_folding/current/datasets/hcp/crops/2mm/CINGULATE/mask/Rcrops/{id}_cropped_skeleton.nii.gz"

#cropi = to_bucket(a.loadObject(mm_skeleton_path))
#windoforcrop.addObjects(cropi)

In [ ]:
def crop_mask(file_src, file_cropped, mask):
    """Crops according to mask"""
    volume = aims.read(file_src)
    print(np.count_nonzero(volume.np))
    if mask:
        mask = aims.read(mask)
        arr = volume.np
        arr_mask = np.asarray(mask)
        arr[arr_mask == 0] = 0
        print(np.count_nonzero(volume.np))
    aims.write(volume, file_cropped)

To prepare te files for the convert_volume_to_bucket for a given brain region

In [ ]:
mask_path = f'/neurospin/dico/data/deep_folding/current/datasets/hcp/crops/2mm/{region}/mask/{side}mask_skeleton.nii.gz'
file_src = f'/neurospin/dico/data/deep_folding/current/datasets/hcp/skeletons/2mm/{side}/{side}resampled_skeleton_{subject}.nii.gz'
file_cropped = f'/volatile/ad279118/Figures_report/{subject}_{region}_{side}_cropped_skeleton.nii.gz'
crop_mask(file_src, file_cropped, mask_path)

In [ ]:
file_src

To do prepare the file to get the buckets for all sulci for on hemisphere for one subject

In [ ]:
mask_path = False
file_src = f'/neurospin/dico/data/deep_folding/current/datasets/hcp/skeletons/2mm/{side}/{side}resampled_skeleton_{subject}.nii.gz'
file_cropped = f'/volatile/ad279118/Figures_report/{subject}_{side}_skeleton.nii.gz'
crop_mask(file_src, file_cropped, mask_path)

To do from a terminal with bv bash so as to get the buckets

In [ ]:
"""
cd /volatile/ad279118/deep_folding/deep_folding/brainvisa/utils
python3 convert_volume_to_bucket.py -s '/volatile/ad279118/Figures_report' -t '/volatile/ad279118/Figures_report'
"""

To visualize in the end the region included in the whole brain (sulci point of view)

In [ ]:
w5 = a.createWindow("3D")
path_to_bck = f"/volatile/ad279118/Figures_report/Cropped_files/{subject}_{region}_{side}_cropped_skeleton.bck"


bcks = a.loadObject(path_to_bck)
bcks.loadReferentialFromHeader()

path_to_source = f"/volatile/ad279118/Figures_report/Cropped_files/{subject}_{side}_skeleton.bck"

source = a.loadObject(path_to_source)
source.loadReferentialFromHeader()

#w5.addObjects(Rwhite)
#w5.addObjects(Rsulci)
w5.addObjects(bcks)
w5.addObjects(source)

In [ ]:
path_to_bck

In [ ]:
path_to_source